In [3]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns 

In [4]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source'],
      dtype='str')

In [5]:
all_data_systems = systems_cleaned[
    systems_cleaned['has_current_data']
    & systems_cleaned['has_irradiance_data']
    & systems_cleaned['has_power_data']
    & systems_cleaned['has_temperature_data']
    & systems_cleaned['has_voltage_data']
    & systems_cleaned['has_ambient_temperature_data']
]
all_data_ids = set(all_data_systems.system_id)

In [6]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [7]:
my_system_ids = list(all_data_ids.intersection(metrics_id_set))
my_system_ids.sort()
num_ids = len(my_system_ids)
num_ids

36

# Early Ideas

In [9]:
system_id = 1416
relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
for j in relevant_rows_systems.index[0:1]:
    max_dc_capacity = relevant_rows_systems.loc[j, 'dc_capacity_kW']
    system_type = relevant_rows_systems.loc[j, 'simplified_type']
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]

In [10]:
relevant_rows_systems

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_ambient_temperature_data,has_temperature_data,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source
143,1416,[1416] GSA Sanford Federal Building,"Raleigh, NC",7,35.7793,78.63411,110.0,564.5,ET,11,...,False,True,True,True,True,True,True,Unknown,unknown,PVDAQ General


In [11]:
relevant_rows_metrics

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
1107,1416,4742,dc_power,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power__4742
1108,1416,4762,ac_current_phA,AC current,A,A,1.0,0.0,,avg,NaN,NaN,,ac_current_pha__4762
1109,1416,4763,ac_current_phB,AC current,A,A,1.0,0.0,,avg,NaN,NaN,,ac_current_phb__4763
1110,1416,4764,ac_current_phC,AC current,A,A,1.0,0.0,,avg,NaN,NaN,,ac_current_phc__4764
1111,1416,4749,ac_current,AC current,A,A,1.0,0.0,,avg,NaN,NaN,,ac_current__4749
1112,1416,4746,power_factor,AC other,-,-,1.0,0.0,,avg,NaN,NaN,,power_factor__4746
1113,1416,4753,power_factor_phA,AC other,-,-,1.0,0.0,,avg,NaN,NaN,,power_factor_pha__4753
1114,1416,4754,power_factor_phB,AC other,-,-,1.0,0.0,,avg,NaN,NaN,,power_factor_phb__4754
1115,1416,4755,power_factor_phC,AC other,-,-,1.0,0.0,,avg,NaN,NaN,,power_factor_phc__4755
1116,1416,4745,apparent_power_kVA,AC other,kVA,VA,1000.0,0.0,,avg,NaN,NaN,,apparent_power_kva__4745


## First search for metrics with fragments in the name

In [12]:
def metrics_search_for_fragment_df(df: pd.DataFrame, fragment: str):
    fragment = fragment.lower()
    return df[
        (df.loc[:, 'sensor_name'].str.contains(fragment, case=False))
        | (df.loc[:, 'common_name'].str.contains(fragment, case=False))
    ]

In [13]:
def metrics_search_for_two_fragments_df(df: pd.DataFrame, fragment_1: str,
                                        fragment_2: str, and_or: str):
    fragment_1 = fragment_1.lower()
    fragment_2 = fragment_2.lower()
    if and_or == 'and':
        return df[
            ((df.loc[:, 'sensor_name'].str.contains(fragment_1, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_1, case=False)))
            & ((df.loc[:, 'sensor_name'].str.contains(fragment_2, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_2, case=False)))
        ]
    elif and_or == 'or':
        return df[
            ((df.loc[:, 'sensor_name'].str.contains(fragment_1, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_1, case=False)))
            | ((df.loc[:, 'sensor_name'].str.contains(fragment_2, case=False))
             | (df.loc[:, 'common_name'].str.contains(fragment_2, case=False)))
        ]

In [14]:
# sample use -- search for dc and power
system_id = 1283
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
metrics_search_for_two_fragments_df(relevant_rows_metrics, 'pow', 'dc', 'and')

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
502,1283,1134,inv1_dc_power,DC power,W,W,1.0,0.0,inv1_dc_voltage*inv1_dc_current,avg,NaN,NaN,,inv1_dc_power__1134
503,1283,1135,inv2_dc_power,DC power,W,W,1.0,0.0,inv2_dc_voltage*inv2_dc_current,avg,NaN,NaN,,inv2_dc_power__1135
504,1283,1136,dc_power,DC power,W,W,1.0,0.0,inv1_dc_power+inv2_dc_power,avg,NaN,NaN,,dc_power__1136


## First listing of DC_power and AC_power aggregate names

In [15]:
def construct_all_parquet_power_aggregator_names():
    global metrics_df, systems_cleaned
    power_data = metrics_search_for_fragment_df(metrics_df, 'pow')
    power_ids = set(power_data.system_id).intersection(set(systems_cleaned.system_id))
    answers_dict = {
        id: [] for id in power_ids
    }
    # manually computed list
    dc_sensor_names = ['dc_power', 'dc_power_hW', 'dc_power_kW', 'dc_power_1_6',
                       'InvPDC_kW_Avg', 'dc_power_calc']
    ac_sensor_names = ['ac_power', 'ac_power_hW', 'ac_power_kW', 'ac_power_1_6',
                       'InvPAC_kW_Avg', 'ac_power_calc', 'W_avg',
                       'ac_power_metered_kW', 'RTW']
    for sensor_name in dc_sensor_names + ac_sensor_names:
        exact_name_metrics = metrics_df[
                metrics_df['sensor_name'] == sensor_name
        ]
        for id in set(exact_name_metrics['system_id']).intersection(set(systems_cleaned.system_id)):
            relevant_rows_metrics = metrics_df[(metrics_df['system_id']==id)
                                               & (metrics_df['sensor_name']==sensor_name)]
            if len(relevant_rows_metrics.index) > 1:
                raise RuntimeError(f'System {id} has multiple sensors named {sensor_name}!')
            else:
                ind = relevant_rows_metrics.index[0]
                metric_id = relevant_rows_metrics.loc[ind, 'metric_id']
                common_name = relevant_rows_metrics.loc[ind, 'common_name']
                given_unit = relevant_rows_metrics.loc[ind, 'units']
                calc_type = relevant_rows_metrics.loc[ind, 'calc_details']
                answers_dict[id].append({
                    'metric_id': metric_id,
                    'sensor_name': sensor_name,
                    'common_name': common_name,
                    'units': given_unit,
                    'calc_details': calc_type
                })
    # quick checks
    for id in power_ids:
        # check for missing entries
        if len(answers_dict[id]) == 0:
            print(f'System {id} appears to have no obvious power aggregator name.')

    return answers_dict

Justify the inclusion of 'W_avg', 'RTW' in ac sensor names!

In [16]:
# show nothing but AC power
assert(set(metrics_df[metrics_df['sensor_name'] == 'W_avg'].common_name) == {'AC power'})
assert(set(metrics_df[metrics_df['sensor_name'] == 'RTW'].common_name) == {'AC power'})

In [17]:
agg_power = construct_all_parquet_power_aggregator_names()

# Towards getting All DC_Power system names

In [ ]:
def sort_all_dc_power_names_step_1():
    '''Find all dc aggregator names.'''
    global metrics_df, systems_cleaned
    power_data = metrics_search_for_fragment_df(metrics_df, 'pow')
    dc_data = metrics_search_for_fragment_df(metrics_df, 'dc')
    power_ids = set(power_data.system_id).intersection(set(systems_cleaned.system_id),
                                                       set(dc_data.system_id))
    # remove 3 false positives -- systems with DC voltage but not DC power
    power_ids = power_ids.difference({1367, 1432, 1433})
    answers_dict = {
        id: [] for id in power_ids
    }
    dc_agg_sensor_names = ['dc_power', 'dc_power_hW', 'dc_power_kW', 'dc_power_1_6',
                       'InvPDC_kW_Avg', 'dc_power_calc']
    for sensor_name in dc_agg_sensor_names:
        exact_name_metrics = metrics_df[
                metrics_df['sensor_name'] == sensor_name
        ]
        for id in set(exact_name_metrics['system_id']).intersection(set(systems_cleaned.system_id)):
            relevant_rows_metrics = metrics_df[(metrics_df['system_id']==id)
                                               & (metrics_df['sensor_name']==sensor_name)]
            if len(relevant_rows_metrics.index) > 1:
                raise RuntimeError(f'System {id} has multiple sensors named {sensor_name}!')
            else:
                ind = relevant_rows_metrics.index[0]
                metric_id = relevant_rows_metrics.loc[ind, 'metric_id']
                common_name = relevant_rows_metrics.loc[ind, 'common_name']
                given_unit = relevant_rows_metrics.loc[ind, 'units']
                calc_type = relevant_rows_metrics.loc[ind, 'calc_details']
                answers_dict[id].append({
                    'metric_id': metric_id,
                    'sensor_name': sensor_name,
                    'common_name': common_name,
                    'units': given_unit,
                    'calc_details': calc_type,
                    'whole_or_part': 'whole'
                })
    # quick checks
    for id in power_ids:
        # check for missing entries
        if len(answers_dict[id]) == 0:
            print(f'System {id} appears to have no obvious DC power aggregator name.')
        # check for duplicates
        elif len(answers_dict[id]) != 1:
            print(f'System {id} has multiple DC power aggregators!')
            for term in answers_dict[id]:
                print(term)

    return answers_dict

Verify that the manually excluded terms have ac power and dc voltage

In [19]:
for id in {1367, 1432, 1433}:
    relevant_rows_metrics = metrics_df[metrics_df['system_id']==id]
    assert('ac_power' in relevant_rows_metrics.sensor_name.unique())
    assert(any(relevant_rows_metrics.sensor_name.str.contains('dc_voltage')))

In [20]:
dc_agg_names = sort_all_dc_power_names_step_1()

In [21]:
my_units = []
for id in dc_agg_names.keys():
    for metric_dict in dc_agg_names[id]:
        my_units.append(metric_dict['units'])
my_units = set(my_units)
my_units


{'W', 'kW'}

In [ ]:
def common_prefix_and_suffix(names_collection, first_name):
    '''Find the common prefix and suffix of a collection of the strings,
    with the first name in the collection set aside for ease of coding.'''
    common_prefix = ''
    j = 0
    good_prefix = True
    while good_prefix:
        if all(
            [name.startswith(common_prefix) for name in names_collection]
        ):
            j += 1
            common_prefix = first_name[0:j]
        else: # bad prefix, back it up one
            good_prefix = False
            common_prefix = common_prefix[0:-1]
    common_suffix = ''
    j = 0 
    good_suffix = True
    while good_suffix:
        if all(
            [name.endswith(common_suffix) for name in names_collection]
        ):
            j += 1
            common_suffix = first_name[-j:]
        else: # take the last amendment off
            good_suffix = False
            common_suffix = common_suffix[1:]
    return (common_prefix, common_suffix)


def sort_all_dc_power_names_step_2():
    '''Add the DC-power partial data to the DC-power aggregate data.'''
    global metrics_df, systems_cleaned
    # grab the aggregate power names from Step 1
    power_names = sort_all_dc_power_names_step_1()
    # prep my series of booleans for having sub-parts
    has_dc_power_subparts = pd.Series(
        {id: False for id in power_names.keys()}, dtype='bool'
    )
    dc_agg_sensor_names = ['dc_power', 'dc_power_hW', 'dc_power_kW', 'dc_power_1_6',
                           'InvPDC_kW_Avg', 'dc_power_calc']
    for id in power_names.keys():
        # grab only those metrics with both dc and power
        relevant_rows_metrics = metrics_df[metrics_df['system_id']==id]
        dc_and_power_metrics = metrics_search_for_two_fragments_df(
            relevant_rows_metrics,
            'dc',
            'pow',
            'and'
        )
        # drop the aggregate names
        dc_power_non_agg_metrics = dc_and_power_metrics[
            ~dc_and_power_metrics['sensor_name'].isin(dc_agg_sensor_names)
        ]
        #see if any terms remaining
        num_subparts = dc_power_non_agg_metrics.shape[0]
        # system_id 1416 is a special case, see below.
        if (num_subparts > 0) and (id != 1416):
            has_dc_power_subparts[id] = True
            # sort remaining names
            dc_power_non_agg_metrics = dc_power_non_agg_metrics.sort_values('sensor_name')
            # determine common prefix and suffix
            dc_power_sensor_names = dc_power_non_agg_metrics['sensor_name'].values
            first_name = dc_power_sensor_names[0]
            common_prefix, common_suffix = common_prefix_and_suffix(dc_power_sensor_names, first_name)
            # add the partial names on there
            for k in range(0, num_subparts):
                kth_metric = dc_power_non_agg_metrics.iloc[k, :]
                # cleaning step -- some sub-parts had no units!
                kth_unit = kth_metric['units']
                if (kth_unit.lower() != 'w') and (kth_unit.lower() != 'kw'):
                    # should only happen with system 1207
                    assert(id == 1207)
                    # grab the unit from the unit in the aggregate collector
                    # should be 'W'
                    kth_unit = power_names[1207][0]['units']
                kth_sensor_name = kth_metric['sensor_name']
                if kth_sensor_name.startswith(common_prefix)\
                  and kth_sensor_name.endswith(common_suffix):
                    kth_interior = kth_sensor_name.removeprefix(common_prefix).removesuffix(common_suffix)
                else:
                   raise ValueError('Bad prefix or suffix!') 
                power_names[id].append({
                    'metric_id': kth_metric['metric_id'],
                    'sensor_name': kth_sensor_name,
                    'common_name': kth_metric['common_name'],
                    'units': kth_unit,
                    'calc_details': kth_metric['calc_details'],
                    'whole_or_part': 'part',
                    'index': kth_interior
                })
        elif id == 1416:
            # show that this case is some positive-negative decomposition
            # which is not really a sub-part in the same way
            assert(set(dc_power_non_agg_metrics.sensor_name)
                   == {'dc_power_positive', 'dc_power_negative'})
            # hence, do not add anything for now.  
    return (power_names, has_dc_power_subparts)
        


In [23]:
dc_power_names, dc_power_subparts = sort_all_dc_power_names_step_2()

In [25]:
# draft of common-prefix and common-suffix code used above
for id in dc_power_names.keys():
    if dc_power_subparts[id]:
        print(id)
        subpart_sensor_names = [dc_power_names[id][j]['sensor_name']
                                for j in range(1, len(dc_power_names[id]))]
        first_sensor_name = dc_power_names[id][1]['sensor_name']
        common_prefix = ''
        j = 0
        good_prefix = True
        while good_prefix:
            if all(
                [name.startswith(common_prefix) for name in subpart_sensor_names]
            ):
                j += 1
                common_prefix = first_sensor_name[0:j]
            else: # bad prefix, back it up one
                good_prefix = False
                common_prefix = common_prefix[0:-1]
        common_suffix = ''
        j = 0
        good_suffix = True
        while good_suffix:
            if all(
                [name.endswith(common_suffix) for name in subpart_sensor_names]
            ):
                j += 1
                common_suffix = first_sensor_name[-j:]
            else: # take the last amendment off
                good_suffix = False
                common_suffix = common_suffix[1:]
        # rule out dc_power_positive and dc_power_negative as being
        # too far apart
        non_common_len = len(first_sensor_name) - len(common_prefix) - len(common_suffix)
        if non_common_len >= 3:
            print('Too many differences')
        else:
            indices = []
            for metric_dict in dc_power_names[id][1:]:
                sensor_name = metric_dict['sensor_name']
                if sensor_name.startswith(common_prefix)\
                    and sensor_name.endswith(common_suffix):
                    interior = sensor_name.removeprefix(common_prefix).removesuffix(common_suffix)
                    indices.append(interior)
                else:
                    print('Bad prefix or suffix!')
            print(indices)

                    

1283
['1', '2']
1305
['1', '2', '3']
1306
['1', '2', '3', '4', '5', '6']
1307
['1', '2', '3']
1308
['1', '2', '3']
1310
['1', '2', '3', '4', '5', '6']
1312
['1', '2', '3']
4901
['1', '2', '3', '4', '5', '6', '7']
4902
['1', '2', '3', '4', '5', '6', '7']
4903
['1', '2', '3', '4']
1199
['1', '2', '3', '4', '5', '6', '7']
1200
['1', '2', '3', '4', '5', '6']
1202
['1', '2', '3', '4', '5', '6']
1203
['1', '2']
1204
['1', '2', '3', '4', '5', '6', '7', '8', '9']
1332
['1', '2', '3']
50
['1', '2']
1207
['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
1208
['1', '2']
51
['1', '2']
1368
['1', '2', '3', '4', '5', '6']
1369
['1', '2', '3', '4', '5', '6', '7', '8', '9']
1278
['1', '2']


In [26]:
units_collection = []
for id in dc_power_names.keys():
    for metric_dict in dc_power_names[id]:
        units_collection.append(metric_dict['units'])
units_set = set(units_collection)
units_set
        

{'W', 'kW'}

In [27]:
dc_power_names[1416]

[{'metric_id': np.int32(4742),
  'sensor_name': 'dc_power',
  'common_name': 'DC power',
  'units': 'W',
  'calc_details': '',
  'whole_or_part': 'whole'}]

In [28]:
dc_power_subparts[1207]

np.True_

In [29]:
def dc_power_total_name(size_standard: str):
    return f'dc_power_whole_{size_standard}'


def dc_power_partial_name(size_standard: str, ind: int):
    return f'dc_power_subsystem_{ind}_{size_standard}'


def dc_power_dataframe_generator(system_id: int, 
                                 tall_or_wide: str,
                                 error_on_no_data: bool,
                                 size_standard: str):
    '''Make the (tall or wide) pandas DataFrame with all dc power data.'''
    dc_power_names, _ = sort_all_dc_power_names_step_2()
    try:
        my_dc_power_names = dc_power_names[system_id]
    except KeyError:
        if error_on_no_data:
            raise ValueError(f'System {system_id} has no DC power data!')
        else:
            return None
    except BaseException as e:
        raise e
    metric_ids = []
    whole_metric_ids = []
    # grab all metric ids, putting the 'whole' category first
    for metric_data_dict in my_dc_power_names:
        if metric_data_dict['whole_or_part'] == 'whole':
            metric_ids.insert(0, metric_data_dict['metric_id'])
            whole_metric_ids.append(metric_data_dict['metric_id'])
        elif metric_data_dict['whole_or_part'] == 'part':
            metric_ids.append(metric_data_dict['metric_id'])
        else:
            raise ValueError('The "whole_or_part" result of sort_all_power_names_part_2()\n'
                             f'is not correct for system {system_id}.')
    # Load only these metrics from the system
    my_system_parquet_data_path = Path(f'../../../../data_ds_project/systems/parquet/{system_id}/')
    my_system_parquet_selection = pq.ParquetDataset(
        my_system_parquet_data_path, filters=[
            ('metric_id', 'in', metric_ids)
        ]
    )
    system_df = my_system_parquet_selection.read().to_pandas()
    # for reference, 4 columns (see
    # https://github.com/openEDI/documentation/blob/main/pvdaq.md#pvdaq_pvdata)
    # measured_on, utc_measured_on, metric_id, value)
    # standard cleaning
    system_df = system_df.drop_duplicates()
    # See if multiple values at a given time
    # if so, forced to replace value by mean value
    if any(system_df.duplicated(subset = ['measured_on', 'metric_id'])):
        system_df.loc[:, 'mean_value'] = system_df.groupby(
            ['measured_on', 'metric_id']
        )['value'].transform('mean')
        system_df = system_df.drop(columns='value')
        system_df = system_df.rename(columns={'mean_value':'value'})
        system_df.drop_duplicates()
    # if still duplicates, forced to drop utc_measured_on,
    # a frequent source of off-by-one-hour errors
    # (and points with the same 'measured_on' but different 'utc_measured_on'
    # have the same value, so it is likely that utc_measured_on is the problem)
    if any(system_df.duplicated(subset = ['measured_on', 'metric_id', 'value'])):
        system_df = system_df.drop(columns='utc_measured_on')
        system_df = system_df.drop_duplicates()
    # ready to widen the columns
    wide_df = system_df.pivot(
        index='measured_on',
        columns='metric_id',
        values='value'
    )
    # reset the metric_id name of the index of columns
    wide_df.columns.name = ''
    # reset the index
    wide_df = wide_df.reset_index()
    # before continuing, standardize the capitalization of the size term
    if size_standard.lower() == 'w':
        size_standard = 'W'
    elif size_standard.lower() == 'kw':
        size_standard = 'kW'
    else:
        raise ValueError('Only supports watts and kilowatts for now.')
    # standardize units -- probably only necessary for power and temperature
    # Irradiance is pretty clearly in W/m^2, current in A, voltage in V
    if size_standard == 'W':
        for metric_data_dict in my_dc_power_names:
            if metric_data_dict['units'].lower() == 'kw':
                wide_df.loc[:, metric_data_dict['metric_id']] = wide_df[metric_data_dict['metric_id']] * 1000
    elif size_standard == 'kW':
        for metric_data_dict in my_dc_power_names:
            if metric_data_dict['units'].lower() == 'w':
                wide_df.loc[:, metric_data_dict['metric_id']] = wide_df[metric_data_dict['metric_id']] / 1000
    else:
        raise ValueError('Only supports watts and kilowatts for now.')
    # push the 'whole' columns to the beginning of the pack
    # despite re-ordering earlier, can still be loaded in the wrong order.
    reordered_columns = ['measured_on'] + whole_metric_ids+ (wide_df.columns.drop(
        ['measured_on'] + whole_metric_ids
    ).tolist())
    wide_df = wide_df[reordered_columns]
    # rename columns
    renamer_dict = dict()
    for metric_data_dict in my_dc_power_names:
        if metric_data_dict['whole_or_part'] == 'whole':
            renamer_dict[metric_data_dict['metric_id']] = dc_power_total_name(size_standard)
        elif metric_data_dict['whole_or_part'] == 'part':
            renamer_dict[metric_data_dict['metric_id']] = dc_power_partial_name(
                size_standard, metric_data_dict['index']
            )
        else:
            raise ValueError('The "whole_or_part" result of sort_all_power_names_part_2()\n'
                             f'is not correct for system {system_id}.')
    wide_df = wide_df.rename(columns=renamer_dict)
    # convert back to tall format if that is what we wanted
    if tall_or_wide == 'wide':
        return wide_df
    elif tall_or_wide == 'tall':
        return wide_df.melt(
            id_vars='measured_on',
            value_vars=list(renamer_dict.values()),
            var_name='metric_name',
            value_name='value'
        )
    else:
        raise ValueError('The term "tall_or_wide" must be "tall" or "wide.\n'
                         + f'Recieved {tall_or_wide}')


### Test

In [30]:
test_1207 = dc_power_dataframe_generator(1207, 'wide', False, 'kW')

In [31]:
test_1207.head()

,measured_on,dc_power_whole_kW,dc_power_subsystem_A1_kW,dc_power_subsystem_A2_kW,dc_power_subsystem_B1_kW,dc_power_subsystem_B2_kW,dc_power_subsystem_C1_kW,dc_power_subsystem_C2_kW
0,2011-04-22 15:40:00,3.0620,0.5329,0.5208,0.5217,0.5264,0.4953,0.4649
1,2011-04-22 15:50:00,5.0174,0.8660,0.8480,0.8590,0.8700,0.8180,0.7564
2,2011-04-22 16:00:00,3.8773,0.6714,0.6546,0.6653,0.6755,0.6355,0.5750
3,2011-04-22 16:10:00,2.9419,0.5111,0.4940,0.5041,0.5152,0.4817,0.4358
4,2011-04-22 16:20:00,2.2972,0.3989,0.3823,0.3936,0.4043,0.3730,0.3451


In [32]:
test_1207['manual_sum'] = test_1207['dc_power_subsystem_A1_kW'] + test_1207['dc_power_subsystem_A2_kW']\
    + test_1207['dc_power_subsystem_B1_kW'] + test_1207['dc_power_subsystem_B2_kW']\
    + test_1207['dc_power_subsystem_C1_kW'] + test_1207['dc_power_subsystem_C2_kW']  

In [33]:
test_1207['diff'] = np.isclose(test_1207['dc_power_whole_kW'].values, test_1207['manual_sum'].values)

In [34]:
test_1207['diff'].value_counts()

diff
True    357816
Name: count, dtype: int64

In [35]:
power_agg_names = sort_all_dc_power_names_step_1()
for id in power_agg_names.keys():
    print(id)
    temp_df = dc_power_dataframe_generator(id, 'wide', True, 'W')
    print(temp_df.columns)

2
Index(['measured_on', 'dc_power_whole_W'], dtype='str', name='')
3
Index(['measured_on', 'dc_power_whole_W'], dtype='str', name='')
1283


KeyboardInterrupt: 

## Testing ground -- Can ignore after this.

In [205]:
path_1283 = Path(f'../../../../data_ds_project/systems/parquet/1283/')
pq_1283 = pq.ParquetDataset(
    path_1283, filters=[
            ('metric_id', 'in', [1136, 1134, 1135])
    ]
)
df_1283 = pq_1283.read().to_pandas()
df_1283 = df_1283.drop_duplicates()
df_1283.tail()

,measured_on,utc_measured_on,metric_id,value
46847841,2022-02-08 18:36:00,2022-02-09 01:36:00,1135,23555.9562
46847842,2022-02-08 18:36:15,2022-02-09 01:36:15,1135,23383.2789
46847843,2022-02-08 18:36:30,2022-02-09 01:36:30,1135,23091.3712
46847844,2022-02-08 18:36:45,2022-02-09 01:36:45,1135,22906.8921
46847845,2022-02-08 18:37:00,2022-02-09 01:37:00,1135,22862.8058


In [110]:
wide_1283 = df_1283.pivot(
    index='measured_on',
    columns='metric_id',
    values='value'
)

In [131]:
wide_1283.head()

,1134,1135,1136
measured_on,,,
2012-01-09 17:00:45,0.0,0.0,0.0
2012-01-09 17:01:00,0.0,0.0,0.0
2012-01-09 17:01:15,0.0,0.0,0.0
2012-01-09 17:01:30,0.0,0.0,0.0
2012-01-09 17:01:45,0.0,0.0,0.0


In [130]:
wide_1283.columns.name = ''

In [111]:
wide_1283.describe()

metric_id,1134,1135,1136
count,1.758535e+07,1.758530e+07,1.161351e+07
mean,1.085359e+05,1.498444e+05,3.729155e+05
std,2.633018e+06,2.706288e+06,6.515954e+06
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,0.000000e+00
75%,0.000000e+00,5.488920e+04,5.365824e+04
max,6.398400e+07,6.398400e+07,1.279680e+08


In [ ]:
df_1283.groupby('metric_id').mean()

,measured_on,utc_measured_on,value
metric_id,,,
1134,2017-01-20 02:49:01.558431488,2020-09-16 08:45:12.218384128,108535.929947
1135,2017-01-20 02:44:39.528924416,2020-09-16 08:48:05.502534400,149844.364386
1136,2015-03-20 10:26:00.958238720,NaT,372915.506077


In [107]:
first_overview = df_1283[df_1283['measured_on'] == '2018-09-19 10:31:15']
first_overview

,measured_on,utc_measured_on,metric_id,value
35255273,2018-09-19 10:31:15,NaT,1134,0.0000
35255319,2018-09-19 10:31:15,NaT,1135,140159.7083


In [216]:
relevant_rows_metrics[relevant_rows_metrics['common_name'].str.contains('pow')]

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
1107,1416,4742,dc_power,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power__4742
1129,1416,4741,dc_power_negative,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power_negative__4741
1130,1416,4738,dc_power_positive,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power_positive__4738
1135,1416,4743,real_power_kW,AC power other,kW,W,1000.0,0.0,,avg,NaN,NaN,,real_power_kw__4743
1136,1416,4750,real_power_phA_kW,AC power other,kW,W,1000.0,0.0,,avg,NaN,NaN,,real_power_pha_kw__4750
1137,1416,4751,real_power_phB_kW,AC power other,kW,W,1000.0,0.0,,avg,NaN,NaN,,real_power_phb_kw__4751
1138,1416,4752,real_power_phC_kW,AC power other,kW,W,1000.0,0.0,,avg,NaN,NaN,,real_power_phc_kw__4752


In [207]:
path_1416= Path(f'../../../../data_ds_project/systems/parquet/1416/')
pq_1416 = pq.ParquetDataset(
    path_1416, filters=[
            ('metric_id', 'in', [4738, 4741, 4742])
    ]
)
df_1416 = pq_1416.read().to_pandas()
df_1416 = df_1416.drop_duplicates()
df_1416.tail()

,measured_on,utc_measured_on,metric_id,value
10525176,2018-03-26 22:50:25,NaT,4741,0.288871
10525177,2018-03-26 22:50:30,NaT,4738,0.140005
10525178,2018-03-26 22:50:30,NaT,4741,0.216653
10525179,2018-03-26 22:50:35,NaT,4738,0.123390
10525180,2018-03-26 22:50:35,NaT,4741,0.361088


In [209]:
df_1416_wide = df_1416.pivot(
    index='measured_on',
    columns = 'metric_id',
    values='value'
)
df_1416_wide.columns.name = ''
df_1416_wide = df_1416_wide.reset_index()
df_1416_wide = df_1416_wide.rename(columns={
    4738: 'dc_power_positive',
    4741: 'dc_power_negative',
    4742: 'dc_power'
})

In [211]:
df_1416_wide

,measured_on,dc_power_positive,dc_power_negative,dc_power
0,2015-11-05 16:33:10,10850.730000,10157.660000,21008.390000
1,2015-11-05 16:33:15,10896.620000,10038.930000,20935.550000
2,2015-11-05 16:33:20,10855.990000,9928.960000,20784.950000
3,2015-11-05 16:33:25,11105.000000,9872.675000,20977.680000
4,2015-11-05 16:33:30,10604.860000,9857.531000,20462.390000
...,...,...,...,...
3508389,2018-03-26 22:50:15,0.140005,-0.288871,-0.148866
3508390,2018-03-26 22:50:20,0.140005,-0.333397,-0.193393
3508391,2018-03-26 22:50:25,0.156619,0.288871,0.445490
3508392,2018-03-26 22:50:30,0.140005,0.216653,0.356658


In [219]:
df_1416_wide['manual_sum'] = df_1416_wide['dc_power_positive'] + df_1416_wide['dc_power_negative']
df_1416_wide['manual_diff'] = df_1416_wide['dc_power_positive'] - df_1416_wide['dc_power_negative']

In [220]:
df_1416_wide['dc_power_negative'].describe()

count    3.508394e+06
mean     1.796907e+04
std      4.455719e+04
min     -1.294736e+03
25%     -1.877968e+01
50%     -1.011047e+00
75%      1.768127e+03
max      2.769924e+05
Name: dc_power_negative, dtype: float64

In [222]:
df_1416_wide.sample(15)

,measured_on,dc_power_positive,dc_power_negative,dc_power,manual_sum,manual_diff
3041498,2016-08-19 14:13:35,547.877000,-686.246300,-138.369300,-138.369300,1234.123300
48446,2015-11-08 11:50:20,113277.400000,110765.800000,224043.300000,224043.200000,2511.600000
1775872,2016-02-20 06:07:50,1.120037,-0.669896,0.450141,0.450141,1.789933
3027683,2016-08-18 19:02:20,444.028500,-497.543900,-53.515440,-53.515400,941.572400
3507040,2016-09-15 12:48:50,622.384800,-728.614800,-106.230000,-106.230000,1350.999600
3369746,2016-09-07 14:07:40,662.747000,-607.650100,55.096920,55.096900,1270.397100
706587,2015-12-18 00:58:00,0.840028,-3.740252,-2.900225,-2.900224,4.580280
3237596,2016-08-30 22:35:10,0.189848,-0.199817,-0.009969,-0.009969,0.389665
904529,2015-12-29 11:53:10,52033.090000,54748.180000,106781.300000,106781.270000,-2715.090000
2751255,2016-04-20 03:25:40,1.400046,-2.264886,-0.864840,-0.864840,3.664932
